# Example 5 Business: Recovery of Alameda Island businesses using R2D damage estimates and third-party infrastructure simulators.

This notebook runs the Alameda Island business resilience simulation. It extends Example 5 with business impact modelling, tracking revenue losses caused by building damage, infrastructure outages, employee availability, local supplier access, and customer base disruption.

Example 5 shows how **pyrecodes** extends NHERI R2D's damage assessment to simulate recovery and integrate third-party infrastructure simulators of water supply systems and transportation systems to assess their interdependencies. Sparse distribution time stepping is used. 

Please refer to the **pyrecodes** [Example 5 page](https://nikolablagojevic.github.io/pyrecodes/html/usage/examples/example_5.html) for further details.

In [ ]:
import os
import webbrowser
import folium
import shapely
import pyproj
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from pyrecodes import main
from pyrecodes.component.r2d_component import R2DRoadway, R2DBridge, R2DTunnel, R2DBuildingWithBusiness
from pyrecodes.resource_distribution_model.residual_demand_traffic_distribution_model import ResidualDemandTrafficDistributionModel


def get_point_square(point_geom, half_side_m):
    """Create a square polygon centered on a point geometry with a given half-side in meters."""
    transformer_to_utm = pyproj.Transformer.from_crs("epsg:4326", "epsg:32610", always_xy=True)
    transformer_to_wgs = pyproj.Transformer.from_crs("epsg:32610", "epsg:4326", always_xy=True)
    utm_x, utm_y = transformer_to_utm.transform(point_geom.x, point_geom.y)
    utm_square = shapely.box(utm_x - half_side_m, utm_y - half_side_m,
                             utm_x + half_side_m, utm_y + half_side_m)
    wgs_coords = [transformer_to_wgs.transform(*coord) for coord in utm_square.exterior.coords]
    return wgs_coords

def get_bridge_square(component):
    """Create a square polygon centered on the bridge's point geometry, sized by deck width."""
    point = shapely.from_wkt(component.geometry)
    half_side_m = component.deck_width * 0.3048 / 2
    return get_point_square(point, half_side_m)

def get_tunnel_square(component):
    """Create a circular polygon centered on the tunnel's point geometry, sized by a fixed radius."""
    point = shapely.from_wkt(component.geometry)
    half_side_m = 150
    return get_point_square(point, half_side_m)

def get_traffic_nodes(system):
    """Extract traffic node locations from the ResidualDemandTrafficDistributionModel's flow_simulator.nodes_df."""
    for resource_name, resource_data in system.resources.items():
        dist_model = resource_data.get('DistributionModel', None)
        if isinstance(dist_model, ResidualDemandTrafficDistributionModel):
            nodes_df = dist_model.flow_simulator.nodes_df
            # nodes_df has columns: node_id, x (lon), y (lat)
            return [(row['node_id'], row['y'], row['x']) for _, row in nodes_df.iterrows()]
    return []


def build_map(snapshot, time_step):
    """Build a folium map from a saved snapshot of component states."""
    roads = snapshot['roads']
    bridges = snapshot['bridges']
    tunnels = snapshot['tunnels']
    buildings = snapshot.get('buildings', [])
    traffic_nodes = snapshot.get('traffic_nodes', [])
    center = snapshot['center']
    center_lat, center_lon = center
    m = folium.Map(location=[center_lat, center_lon], zoom_start=14)

    for name, coords, func_level in roads:
        color = 'green' if func_level >= 1.0 else 'red'
        folium.PolyLine(
            coords, color=color, weight=4, opacity=0.8,
            tooltip=f"{name} | Functionality: {func_level:.2f}"
        ).add_to(m)

    for name, folium_coords, func_level in bridges:
        color = 'green' if func_level >= 1.0 else 'red'
        folium.Polygon(
            folium_coords, color=color, fill=True, fill_color=color,
            fill_opacity=0.6, weight=2,
            tooltip=f"{name} | Functionality: {func_level:.2f}"
        ).add_to(m)

    for name, folium_coords, func_level in tunnels:
        color = 'green' if func_level >= 1.0 else 'red'
        folium.Polygon(
            folium_coords, color=color, fill=True, fill_color=color,
            fill_opacity=0.6, weight=2,
            tooltip=f"{name} | Functionality: {func_level:.2f}"
        ).add_to(m)

    for node_id, lat, lon in traffic_nodes:
        folium.CircleMarker(
            location=[lat, lon], radius=3,
            color='gray', fill=True, fill_color='gray',
            fill_opacity=0.5, weight=1,
            tooltip=f"Traffic node {node_id}"
        ).add_to(m)

    for building_record in buildings:
        building_name, footprint_coords, tooltip_text = building_record[0], building_record[1], building_record[2]
        # 4th element (when present) flags a building whose business is cut off from employees,
        # suppliers, or customers at this time step. Colour it red; operating buildings stay blue.
        inaccessible = building_record[3] if len(building_record) > 3 else False
        building_color = 'red' if inaccessible else 'blue'
        folium.Polygon(
            footprint_coords, color=building_color, fill=True, fill_color=building_color,
            fill_opacity=0.7 if inaccessible else 0.4, weight=1,
            tooltip=folium.Tooltip(tooltip_text, sticky=True)
        ).add_to(m)

    folium.map.Marker(
        [center_lat, center_lon],
        icon=folium.DivIcon(
            html=f'<div style="font-size:14px;font-weight:bold;background:white;padding:4px;border-radius:4px;">Time step: {time_step}</div>'
        )
    ).add_to(m)

    legend_html = '''<div style="position:fixed;bottom:30px;left:30px;z-index:9999;background:white;
padding:8px 12px;border:1px solid #999;border-radius:5px;font:13px sans-serif">
<b>Business buildings</b><br>
<span style="color:blue">&#9632;</span> operating<br>
<span style="color:red">&#9632;</span> cut off (0 employees, suppliers, or customers)
</div>'''
    m.get_root().html.add_child(folium.Element(legend_html))
    return m


def capture_snapshot(system, time_step, traffic_nodes):
    """Capture current road/bridge/tunnel/building state as lightweight data for later rendering."""
    road_components = [c for c in system.components if isinstance(c, R2DRoadway)]
    bridge_components = [c for c in system.components if isinstance(c, R2DBridge)]
    tunnel_components = [c for c in system.components if isinstance(c, R2DTunnel)]
    business_buildings = [c for c in system.components if isinstance(c, R2DBuildingWithBusiness)]

    if not road_components and not bridge_components and not tunnel_components:
        return None

    first_comp = road_components[0] if road_components else (bridge_components[0] if bridge_components else tunnel_components[0])
    first_geom = shapely.from_wkt(first_comp.geometry)
    center = (first_geom.centroid.y, first_geom.centroid.x)

    roads = []
    for c in road_components:
        line = shapely.from_wkt(c.geometry)
        coords = [(lat, lon) for lon, lat in line.coords]
        roads.append((c.name, coords, c.functionality_level))

    bridges = []
    for c in bridge_components:
        square_coords = get_bridge_square(c)
        folium_coords = [(lat, lon) for lon, lat in square_coords]
        bridges.append((c.name, folium_coords, c.functionality_level))

    tunnels = []
    for c in tunnel_components:
        square_coords = get_tunnel_square(c)
        folium_coords = [(lat, lon) for lon, lat in square_coords]
        tunnels.append((c.name, folium_coords, c.functionality_level))

    # Reasons whose level 0 means the business has lost all of that input at this time step.
    CUT_OFF_REASONS = ('Labor', 'LocalSuppliers', 'Customer Base')
    buildings = []
    for c in business_buildings:
        if hasattr(c, 'footprint') and hasattr(c, 'businesses'):
            geojson_coords = c.footprint['geometry']['coordinates'][0]
            footprint_coords = [(lat, lon) for lon, lat in geojson_coords]
            tooltip_lines = [f"<b>{c.name}</b><br>"]
            building_inaccessible = False
            for biz in c.businesses:
                reasons = biz.reason_for_drop.get(time_step, [])
                # A business is "cut off" if employee availability, supplier access, or customer
                # base has dropped to 0 (no employees / suppliers / customers can reach it).
                zero_reasons = [r['Name'] for r in reasons
                                if r['Name'] in CUT_OFF_REASONS and r['Level'] <= 0]
                if zero_reasons:
                    building_inaccessible = True
                if reasons:
                    reason_strs = [f"{r['Name']}: {r['Level']:.2f}" for r in reasons]
                    cut = f" [CUT OFF: {', '.join(zero_reasons)}]" if zero_reasons else ''
                    tooltip_lines.append(f"Business {biz.business_id}: {', '.join(reason_strs)}{cut}")
                else:
                    tooltip_lines.append(f"Business {biz.business_id}: No drop")
            tooltip_text = '<br>'.join(tooltip_lines)
            buildings.append((c.name, footprint_coords, tooltip_text, building_inaccessible))

    return {
        'roads': roads, 'bridges': bridges, 'tunnels': tunnels,
        'buildings': buildings, 'traffic_nodes': traffic_nodes, 'center': center
    }


def show_slider(snapshots, out_dir='road_network_maps', open_browser=True):
    """Write per-time-step folium maps plus a standalone HTML slider page, and open it in the browser.

    The previous version rendered the map inside an ipywidgets.Output widget. VS Code (and an
    untrusted JupyterLab) sanitize widget HTML and strip folium's <iframe srcdoc>/<script>, leaving
    only folium's grey "Make this Notebook Trusted to load map" placeholder. Writing the maps to disk
    and opening them in a real browser tab sidesteps notebook output sanitization and trust entirely.
    Returns the path to the generated index.html.
    """
    time_steps = sorted(snapshots.keys())
    if not time_steps:
        print('No snapshots to display.')
        return None

    out_dir = os.path.abspath(out_dir)
    os.makedirs(out_dir, exist_ok=True)

    for ts in time_steps:
        build_map(snapshots[ts], ts).save(os.path.join(out_dir, f'map_{ts}.html'))

    steps_js = ','.join(str(ts) for ts in time_steps)
    index_html = f"""<!DOCTYPE html>
<html><head><meta charset="utf-8"><title>Road network over time</title>
<style>
  body {{ margin:0; font-family:sans-serif; }}
  #bar {{ padding:8px 12px; background:#f4f4f4; border-bottom:1px solid #ddd; }}
  #frame {{ border:none; width:100vw; height:calc(100vh - 50px); }}
  #sld {{ width:60%; vertical-align:middle; }}
</style></head>
<body>
  <div id="bar">Time step: <b id="lbl"></b>
    <input id="sld" type="range" min="0" max="{len(time_steps) - 1}" value="{len(time_steps) - 1}" step="1">
  </div>
  <iframe id="frame"></iframe>
  <script>
    var steps = [{steps_js}];
    var sld = document.getElementById('sld'),
        lbl = document.getElementById('lbl'),
        frame = document.getElementById('frame');
    function upd() {{ var ts = steps[+sld.value]; lbl.textContent = ts; frame.src = 'map_' + ts + '.html'; }}
    sld.addEventListener('input', upd);
    upd();
  </script>
</body></html>"""

    index_path = os.path.join(out_dir, 'index.html')
    with open(index_path, 'w') as f:
        f.write(index_html)

    print(f'Wrote {len(time_steps)} maps to {out_dir}')
    print(f'Open this in a browser: file://{index_path}')
    if open_browser:
        webbrowser.open('file://' + index_path)
    return index_path


def run_with_road_plots(config_file):
    """Run pyrecodes simulation and show a post-simulation slider with spatial plots."""
    input_dict = main.read_json_file(config_file)
    system = main.create_system(input_dict)
    snapshots = {}

    # Extract traffic nodes once (they don't change during simulation)
    traffic_nodes = get_traffic_nodes(system)

    for system.time_step in range(system.START_TIME_STEP, system.MAX_TIME_STEP):
        print(f"Time step: {system.time_step}")

        if system.recovery_target_met():
            system.FINISH = True

        if system.time_step == system.DISASTER_TIME_STEP:
            system.set_initial_damage()

        system.update()
        system.distribute_resources()
        system.update_resilience_calculators()

        snapshot = capture_snapshot(system, system.time_step, traffic_nodes)
        if snapshot is not None:
            snapshots[system.time_step] = snapshot

        if system.time_step > system.DISASTER_TIME_STEP:
            system.recover()

        if system.FINISH:
            print('Resilience assessment finished.')
            break

    # Show interactive slider after simulation completes
    show_slider(snapshots)

    system.road_network_snapshots = snapshots
    return system

In [ ]:
system = run_with_road_plots('./Example 5_business/Alameda_Main.json')

system.calculate_resilience()

In [ ]:
import random
import json

from pyrecodes.plotter.business_plotter import BusinessPlotter

business_plotter = BusinessPlotter()
business_resilience_calculator = system.resilience_calculators[-1]

for business_to_plot in random.sample(business_resilience_calculator.businesses, 15):
    reasons_for_drop = business_plotter.get_reasons_for_drop(business_to_plot)
    reasons_as_lines = business_plotter.get_reasons_for_drop_as_lines(reasons_for_drop)
    business_plotter.plot_business_revenue(business_resilience_calculator.business_revenue[business_to_plot], business_to_plot, save_fig=False, show_fig=True)
    # business_plotter.plot_business_revenue_reasons_for_drop_lines(business_to_plot, reasons_as_lines, reasons_to_plot=['Home Component Functionality'], save_fig=False, show_fig=True, linestyle='-')
    # business_plotter.plot_business_revenue_reasons_for_drop_lines(business_to_plot, reasons_as_lines, reasons_to_plot=['Home Component Functionality', 'Customer Base'], save_fig=False, show_fig=True, linestyle='-')
    # business_plotter.plot_business_revenue_reasons_for_drop_lines(business_to_plot, reasons_as_lines, reasons_to_plot=['Home Component Functionality', 'Customer Base', 'LocalSuppliers'], save_fig=False, show_fig=True, linestyle='-')
    # business_plotter.plot_business_revenue_reasons_for_drop_lines(business_to_plot, reasons_as_lines, reasons_to_plot=['Home Component Functionality', 'Customer Base', 'LocalSuppliers', 'Labor'], save_fig=False, show_fig=True, linestyle='-')
    # business_plotter.plot_business_revenue_reasons_for_drop_lines(business_to_plot, reasons_as_lines, reasons_to_plot=['Home Component Functionality', 'Customer Base', 'LocalSuppliers', 'Labor', 'Infrastructure'], save_fig=False, show_fig=True, linestyle='-')
    business_plotter.plot_business_revenue_reasons_for_drop_lines(business_to_plot, reasons_as_lines, save_fig=False, show_fig=True, linestyle='-')
    # business_plotter.plot_business_gantt_chart(business_to_plot, save_fig=False, show_fig=True)

In [ ]:
total_revenue = business_plotter.calculate_total_revenue(business_resilience_calculator)
business_plotter.plot_total_revenue(total_revenue, show_fig=True, save_fig=False)
business_plotter.plot_total_revenue_BI_CBI(total_revenue, business_resilience_calculator.total_BI, business_resilience_calculator.total_CBI, show_fig=True, save_fig=False)


In [ ]:
total_reasons_for_drop = business_plotter.get_total_reasons_for_drop(business_resilience_calculator.businesses)
total_reasons_as_lines = business_plotter.get_total_reasons_for_drop_as_lines(total_reasons_for_drop)

business_plotter.plot_total_revenue_reasons_for_drop_lines(total_reasons_as_lines, save_fig=False, show_fig=True)